In [ ]:
!pip install segmentation-models-pytorch albumentations tqdm

# Dataset

## image preprocessing

In [ ]:
import cv2
import numpy as np
import os

def preprocess_retina_image(image_path, target_size=(512, 512)):
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"faild read image: {image_path}")
    # 1.
    green_channel = img[:, :, 1]
    
    # 2.  CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced_green = clahe.apply(green_channel)
    
    # 3. resize
    resized_img = cv2.resize(enhanced_green, target_size, interpolation=cv2.INTER_CUBIC)
    
    # 4. 
    final_img = np.stack([resized_img] * 3, axis=-1)
    return final_img

def preprocess_mask(mask_path, target_size=(512, 512)):
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        cap = cv2.VideoCapture(mask_path)
        ret, frame = cap.read()
        if ret:
            mask = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        cap.release()
        
    if mask is None:
        raise FileNotFoundError(f"Can Not read the Mask: {mask_path}")
        
    resized_mask = cv2.resize(mask, target_size, interpolation=cv2.INTER_NEAREST)
    
    _, binary_mask = cv2.threshold(resized_mask, 127, 1, cv2.THRESH_BINARY)
    return binary_mask

def process_dataset(base_path, dataset_name):
    
    input_dir = os.path.join(base_path, dataset_name)
    output_dir = os.path.join(base_path, f"{dataset_name}_512")
    
    print(f"\n🚀 بدء معالجة مجموعة البيانات: {dataset_name}")
    print(f"📁 سيتم الحفظ في: {output_dir}")

    for root, dirs, files in os.walk(input_dir):
        rel_path = os.path.relpath(root, input_dir)
        current_out_dir = os.path.join(output_dir, rel_path)
        
        os.makedirs(current_out_dir, exist_ok=True)
        
        folder_name = os.path.basename(root).lower()
        is_mask = 'mask' in folder_name or 'manual' in folder_name
        
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.gif')):
                in_file_path = os.path.join(root, file)
                
                try:
                    if is_mask:
                        processed = preprocess_mask(in_file_path, target_size=(512, 512))
                        
                        out_file_name = os.path.splitext(file)[0] + '.png'
                        out_file_path = os.path.join(current_out_dir, out_file_name)
                        
                        cv2.imwrite(out_file_path, processed * 255)
                    else:
                        processed = preprocess_retina_image(in_file_path, target_size=(512, 512))
                        out_file_path = os.path.join(current_out_dir, file)
                        cv2.imwrite(out_file_path, processed)
                        
                    print(f"Done: {os.path.join(rel_path, file)}")
                except Exception as e:
                    print(f"Error {file}: {e}")


BASE_DATA_PATH = r"E:\DRP\data"

DATASETS_TO_PROCESS = [
    "DRIVE",
    "High_Resolution_Fundus"
]

for dataset in DATASETS_TO_PROCESS:
    process_dataset(BASE_DATA_PATH, dataset)
    
print("\n🎉 تمت معالجة جميع المجموعات بنجاح!")


🚀 بدء معالجة مجموعة البيانات: DRIVE
📁 سيتم الحفظ في: E:\DRP\data\DRIVE_512
✅ تم: test\images\01_test.tif
✅ تم: test\images\02_test.tif
✅ تم: test\images\03_test.tif
✅ تم: test\images\04_test.tif
✅ تم: test\images\05_test.tif
✅ تم: test\images\06_test.tif
✅ تم: test\images\07_test.tif
✅ تم: test\images\08_test.tif
✅ تم: test\images\09_test.tif
✅ تم: test\images\10_test.tif
✅ تم: test\images\11_test.tif
✅ تم: test\images\12_test.tif
✅ تم: test\images\13_test.tif
✅ تم: test\images\14_test.tif
✅ تم: test\images\15_test.tif
✅ تم: test\images\16_test.tif
✅ تم: test\images\17_test.tif
✅ تم: test\images\18_test.tif
✅ تم: test\images\19_test.tif
✅ تم: test\images\20_test.tif
✅ تم: test\mask\01_test_mask.gif
✅ تم: test\mask\02_test_mask.gif
✅ تم: test\mask\03_test_mask.gif
✅ تم: test\mask\04_test_mask.gif
✅ تم: test\mask\05_test_mask.gif
✅ تم: test\mask\06_test_mask.gif
✅ تم: test\mask\07_test_mask.gif
✅ تم: test\mask\08_test_mask.gif
✅ تم: test\mask\09_test_mask.gif
✅ تم: test\mask\10_test_mas

## data augmentation

In [ ]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

# 1. Augmentation
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.06, scale_limit=0.1, rotate_limit=45, p=0.5),
    A.GridDistortion(p=0.3), 
    A.Normalize(mean=(0.6129, 0.2364, 0.1281), std=(0.3016, 0.1446, 0.0837)), !
    ToTensorV2()
])

valid_transform = A.Compose([
    A.Normalize(mean=(0.6129, 0.2364, 0.1281), std=(0.3016, 0.1446, 0.0837)),
    ToTensorV2()
])

class RetinaDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        
        self.images = sorted([f for f in os.listdir(images_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif'))])
        self.masks = sorted([f for f in os.listdir(masks_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif'))])
        
        assert len(self.images) == len(self.masks), \
            f"❌ خطأ: عدد الصور ({len(self.images)}) لا يطابق عدد الأقنعة ({len(self.masks)})!"
            
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.images_dir, self.images[idx])
        mask_path = os.path.join(self.masks_dir, self.masks[idx]) # الاعتماد على نفس الترتيب
        
        image = cv2.imread(img_path)
        if image is None:
            raise FileNotFoundError(f"❌ تعذر قراءة الصورة: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise FileNotFoundError(f"❌ تعذر قراءة الماسك من المسار: {mask_path}. تأكد من صحة الاسم والامتداد!")
            
        mask = mask.astype(np.float32) / 255.0  
        mask = np.expand_dims(mask, axis=-1)  
        
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
            mask = mask.permute(2, 0, 1).float()
            
        return image, mask


c:\Users\MOHAMMED_PC\miniconda3\envs\myenv\Lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


## read dataset

In [ ]:
from torch.utils.data import ConcatDataset, DataLoader

BASE_PATH = r"../data/"

drive_train_images = os.path.join(BASE_PATH, "DRIVE_512", "training", "images")
drive_train_masks = os.path.join(BASE_PATH, "DRIVE_512", "training", "1st_manual") 
print(drive_train_masks)
drive_dataset = RetinaDataset(
    images_dir=drive_train_images,
    masks_dir=drive_train_masks,
    transform=train_transform
)

hrf_images = os.path.join(BASE_PATH, "High_Resolution_Fundus_512", "images")
hrf_masks = os.path.join(BASE_PATH, "High_Resolution_Fundus_512", "manual1") 
print(hrf_masks)

hrf_dataset = RetinaDataset(
    images_dir=hrf_images,
    masks_dir=hrf_masks,
    transform=train_transform
)

full_train_dataset = ConcatDataset([drive_dataset, hrf_dataset])

print(f"📊 عدد صور DRIVE للتدريب: {len(drive_dataset)}")
print(f"📊 عدد صور HRF للتدريب: {len(hrf_dataset)}")
print(f"📦 إجمالي عدد الصور الكلي الجاهز للتدريب: {len(full_train_dataset)}")

train_loader = DataLoader(
    full_train_dataset, 
    batch_size=2,        
    shuffle=True,        
    num_workers=0,
    drop_last=True ,
    pin_memory=True 
)

../data/DRIVE_512\training\1st_manual
../data/High_Resolution_Fundus_512\manual1
📊 عدد صور DRIVE للتدريب: 20
📊 عدد صور HRF للتدريب: 45
📦 إجمالي عدد الصور الكلي الجاهز للتدريب: 65


# Model

## init model

In [ ]:
import segmentation_models_pytorch as smp
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" {device}")

# 1. بناء U-Net مع Transfer Learning
model = smp.Unet(
    encoder_name="efficientnet-b3", 
    encoder_weights="imagenet",     
    in_channels=3,                  
    classes=1,                      
    activation=None                 # سنتركها None لأننا سنستخدم BCEWithLogitsLoss
).to(device)

# 2. دالة الخسارة الهجينة (Hybrid Loss)
# دمج Dice (ممتاز للشعيرات الدقيقة) مع BCE (ممتاز للتصنيف العام)
dice_loss = smp.losses.DiceLoss(smp.losses.BINARY_MODE, from_logits=True)
bce_loss = nn.BCEWithLogitsLoss()

def criterion(pred, target):
    return dice_loss(pred, target) + bce_loss(pred, target)

# 3. المحسن (Optimizer)
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

✅ سيتم التدريب باستخدام: cuda


## training

In [ ]:
from tqdm.notebook import tqdm

epochs = 10 
best_loss = float('inf')

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    
    for images, masks in pbar:
        images = images.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, masks)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        pbar.set_postfix({'Loss': f"{loss.item():.4f}"})
        
    avg_train_loss = train_loss / len(train_loader)
    print(f"🎯 متوسّط خسارة التدريب للـ Epoch {epoch+1}: {avg_train_loss:.4f}")
    
    # حفظ أفضل نموذج
    if avg_train_loss < best_loss:
        best_loss = avg_train_loss
        torch.save(model.state_dict(), "best_unet_vessels.pth")
        print("💾 تم حفظ أفضل نموذج!")



Epoch 1/10:   0%|          | 0/32 [00:00<?, ?it/s]

🎯 متوسّط خسارة التدريب للـ Epoch 1: 1.2848
💾 تم حفظ أفضل نموذج!


Epoch 2/10:   0%|          | 0/32 [00:00<?, ?it/s]

🎯 متوسّط خسارة التدريب للـ Epoch 2: 0.8399
💾 تم حفظ أفضل نموذج!


Epoch 3/10:   0%|          | 0/32 [00:00<?, ?it/s]

...

Epoch 6/10:   0%|          | 0/32 [00:00<?, ?it/s]

🎯 متوسّط خسارة التدريب للـ Epoch 6: 0.4866
💾 تم حفظ أفضل نموذج!


Epoch 7/10:   0%|          | 0/32 [00:00<?, ?it/s]

🎯 متوسّط خسارة التدريب للـ Epoch 7: 0.4778
💾 تم حفظ أفضل نموذج!


Epoch 8/10:   0%|          | 0/32 [00:00<?, ?it/s]

# predict all folder

In [ ]:
import os
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import segmentation_models_pytorch as smp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = smp.Unet(
    encoder_name="efficientnet-b3", 
    encoder_weights=None, 
    in_channels=3,                  
    classes=1,                      
    activation=None                 
).to(device)

weights_path = "best_unet_vessels.pth"
if os.path.exists(weights_path):
    model.load_state_dict(torch.load(weights_path, map_location=device))
    model.eval()
else:
    raise FileNotFoundError(f"NO model found{weights_path}")


TEST_IMAGES_DIR = r"../data/DRIVE_512/test/images"  
OUTPUT_PREDICTIONS_DIR = r"../data/DRIVE_512/test/predictions_output"

os.makedirs(OUTPUT_PREDICTIONS_DIR, exist_ok=True)

test_files = sorted([f for f in os.listdir(TEST_IMAGES_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif'))])
print(f"📦 تم العثور على {len(test_files)} صورة للاختبار البصري.")

#  Normalization 
mean = np.array([0.6129, 0.2364, 0.1281], dtype=np.float32)
std = np.array([0.3016, 0.1446, 0.0837], dtype=np.float32)

with torch.no_grad():
    for file_name in tqdm(test_files, desc="جاري التنبؤ وحفظ الصور"):
        img_path = os.path.join(TEST_IMAGES_DIR, file_name)
        
        orig_img = cv2.imread(img_path)
        if orig_img is None:
            continue
        orig_resized = cv2.resize(orig_img, (512, 512), interpolation=cv2.INTER_CUBIC)
        
        green_channel = orig_img[:, :, 1]
        clahe = cv2.createCACHE(clipLimit=2.0, tileGridSize=(8, 8)) if hasattr(cv2, 'createCACHE') else cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        enhanced_green = clahe.apply(green_channel)
        resized_green = cv2.resize(enhanced_green, (512, 512), interpolation=cv2.INTER_CUBIC)
        
        img_tensor = resized_green.astype(np.float32) / 255.0
        img_tensor = np.stack([img_tensor] * 3, axis=-1)
        img_tensor = (img_tensor - mean) / std
        img_tensor = torch.from_numpy(img_tensor).permute(2, 0, 1).unsqueeze(0).to(device)
        
        output = model(img_tensor)
        pred_mask = (torch.sigmoid(output) > 0.5).cpu().numpy()[0, 0].astype(np.uint8)
        
        green_mask = np.zeros_like(orig_resized)
        green_mask[pred_mask == 1] = [0, 255, 0]  
        overlay = cv2.addWeighted(orig_resized, 0.7, green_mask, 0.3, 0)
        
        out_path = os.path.join(OUTPUT_PREDICTIONS_DIR, f"pred_{file_name}")
        cv2.imwrite(out_path, overlay)

print(f"🎉 انتهت العملية! يمكنك الآن فتح المجلد الموضح وتصفح النتائج الطبية المذهلة:\n📁 {OUTPUT_PREDICTIONS_DIR}")

💾 تم تحميل أوزان النموذج بنجاح!
📦 تم العثور على 20 صورة للاختبار البصري.


جاري التنبؤ وحفظ الصور:   0%|          | 0/20 [00:00<?, ?it/s]

🎉 انتهت العملية! يمكنك الآن فتح المجلد الموضح وتصفح النتائج الطبية المذهلة:
📁 ../data/DRIVE_512/test/predictions_output
